# Week 17: Use a Workflow When the Steps Are Known

This notebook follows the reviewed Week 17 presentation. Read each concept and calculate the worked example before running its code.

## Lesson map

1. Use a Workflow When the Steps Are Known
2. State, Nodes, and Edges Define a Graph
3. A Node Should Perform One Inspectable Job
4. Edges Make Routing Explicit
5. Tools Turn Model Choices into Proposed Actions
6. The Agent Loop Is Action, Observation, and Decision
7. Checkpoints Make State Durable
8. Retries Must Be Node-Specific and Resume-Safe
9. Human Approval Is a State Transition
10. Stopping Conditions Bound Time, Cost, and Risk
11. A Minimal LangGraph Is Explicit Python
12. Guided Lab: Build a Bounded Research Workflow

Use the same reasoning loop throughout: **predict, run, inspect, explain**.


## 1. Use a Workflow When the Steps Are Known

A **workflow** follows declared steps and branches.

An **agent** uses a model to choose among allowed actions based on state and observations.

Use deterministic code when the next step is known. Grant model choice only where it adds value, such as choosing which read-only search tool can answer an unfamiliar question.

More autonomy increases evaluation, permission, cost, and stopping requirements.

### Work it out first

Invoice processing:

`validate -> extract -> human review -> save`

These steps are known, so a workflow is appropriate.

Research assistant:

The model may choose policy search or product search based on the question, within a bounded tool set.

### Notebook bridge

The LangGraph quickstart shows both a workflow and a tool-using agent.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
if request.type == "invoice":
    next_step = "extract"
else:
    next_step = "clarify"

Expected output:

```text
An explicit branch without an unnecessary model decision.
```


## 2. State, Nodes, and Edges Define a Graph

**State** is the typed record describing the current run.

A **node** is a function that reads state and returns state updates.

An **edge** determines which node runs next.

`START` represents graph input and `END` represents termination.

Store raw, reusable data in state rather than only formatted prose. Every field should have an owner and meaning.

### Work it out first

State:

```text
question, retrieved_docs, draft, approved, attempts
```

Node `retrieve` adds documents. Node `draft` adds an answer. An edge routes to review if approval is required.

### Notebook bridge

The notebook defines state before adding nodes and edges.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
class State(TypedDict):
    question: str
    documents: list[Document]
    answer: str | None
    attempts: int

Expected output:

```text
A typed state schema shared by graph nodes.
```


## 3. A Node Should Perform One Inspectable Job

A good node has:

- declared state inputs;
- one responsibility;
- typed outputs;
- bounded external calls;
- visible errors;
- trace span;
- retry policy only when appropriate.

Small nodes make failures and approvals easier to locate. Do not place retrieval, generation, tool execution, and database write inside one opaque node.

### Work it out first

Node `retrieve_policy`:

Input: question and user authorization  
Action: authorized retrieval  
Output: ranked documents and retrieval metadata  
Failure: denied, timeout, or no evidence

It does not generate an answer.

### Notebook bridge

Learners inspect each quickstart node as a state transformation.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
def retrieve_policy(state: State):
    docs = retriever.invoke(state["question"])
    return {"documents": docs}

Expected output:

```text
A state update containing documents while preserving other state fields.
```


## 4. Edges Make Routing Explicit

A fixed edge always routes to the same node.

A **conditional edge** inspects state and chooses from declared destinations.

Routing functions should return only allowed routes and should be tested independently. Important policy decisions, such as whether a user can write data, should be deterministic code rather than model preference.

### Work it out first

After retrieval:

- documents found -> `draft_answer`
- no documents -> `insufficient_evidence`
- authorization denied -> `denied`

All three outcomes are explicit terminal or next states.

### Notebook bridge

The quickstart workflow adds conditional edges after node results.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
def route_after_retrieval(state: State):
    if state["denied"]:
        return "denied"
    return "draft" if state["documents"] else "insufficient"

Expected output:

```text
One allowed node name determined from typed state.
```


## 5. Tools Turn Model Choices into Proposed Actions

An agent model may propose:

- tool name;
- structured arguments;
- reason reflected in its current messages.

The application:

1. parses arguments;
2. authorizes the tool and scope;
3. applies time and resource limits;
4. executes;
5. records the observation in state;
6. routes to the next decision.

Tool results are untrusted external data and may contain errors or injection text.

### Work it out first

Agent proposes `search_policy(query="refund annual plan")`.

Application checks that search is read-only and tenant-filtered, then stores returned chunks and provenance. It rejects a proposed `delete_policy` because that tool is not in the allowed set.

### Notebook bridge

The agent quickstart alternates a model node with a tool node.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
args = SearchArgs.model_validate(tool_call.args)
authorize(user, "search_policy", args)
observation = search_policy(**args.model_dump())

Expected output:

```text
A bounded observation or a structured denied/error result.
```


## 6. The Agent Loop Is Action, Observation, and Decision

A tool-using agent loop:

1. inspect goal and current state;
2. choose an allowed action or finish;
3. execute through application controls;
4. receive an observation;
5. update state;
6. choose again.

The observation may resolve the question, reveal missing information, or produce an error.

Every loop needs an explicit route to `END`.

### Work it out first

Step 1: search policy -> no matching chunk  
Step 2: search product documentation -> supporting chunk found  
Step 3: answer with citation -> finish

Total tool calls `2`, model decisions `3`.

### Notebook bridge

Learners trace every loop iteration in the LangGraph agent example.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
def should_continue(state):
    last = state["messages"][-1]
    return "tools" if last.tool_calls else END

Expected output:

```text
The graph routes to tools only when the latest model output proposes a call.
```


## 7. Checkpoints Make State Durable

A **checkpoint** is a saved snapshot of graph state at an execution boundary.

Checkpoints support:

- recovery after failure;
- multi-turn memory;
- human approval pauses;
- replay and debugging;
- resuming long-running work.

A **thread ID** identifies which persisted state to load. Access to checkpoints must be authorized because state may contain sensitive data.

### Work it out first

Graph completes retrieve and draft, then pauses for approval.

Checkpoint stores question, documents, draft, and status. The process can restart and resume from that state instead of rerunning retrieval and generation.

### Notebook bridge

Learners add an in-memory checkpointer for the lab and document production storage requirements.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
config = {"configurable": {"thread_id": "request-123"}}
result = graph.invoke(initial_state, config)

Expected output:

```text
State snapshots associated with request-123 in the configured checkpointer.
```


## 8. Retries Must Be Node-Specific and Resume-Safe

Retry only a node whose operation can safely be repeated.

**Idempotent** means repeating the same operation has the same intended effect as running it once.

- read-only search is often retryable;
- charging a card or sending an email may duplicate side effects;
- model calls may produce different outputs on retry;
- validation failures need correction, not blind retry.

Record attempts and cap them.

### Work it out first

Search times out: retry up to `3` attempts.  
Email send times out after the server may have accepted it: check an idempotency key or delivery status before retrying.

### Notebook bridge

The lab simulates a transient search failure and confirms attempt count.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
workflow.add_node(
    "search",
    search_node,
    retry_policy=RetryPolicy(max_attempts=3),
)

Expected output:

```text
Only the search node receives the bounded retry policy.
```


## 9. Human Approval Is a State Transition

Use human approval before high-impact or uncertain actions.

The graph should:

1. prepare a proposal;
2. persist state;
3. interrupt;
4. show evidence and proposed action;
5. accept approve, reject, or edited input;
6. validate human input;
7. resume from the checkpoint;
8. record reviewer and decision.

Approval must occur before the side effect.

### Work it out first

Agent drafts an email and proposed recipients. The graph pauses. Reviewer removes one recipient and approves the edited draft. Only then may the send node execute.

### Notebook bridge

Learners add one approval interrupt to a write-capable mock action.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
decision = interrupt({
    "draft": state["draft"],
    "recipients": state["recipients"],
})

Expected output:

```text
The graph pauses with JSON-serializable review data until resumed.
```


## 10. Stopping Conditions Bound Time, Cost, and Risk

Set:

- maximum graph steps;
- maximum attempts per node;
- maximum tool calls;
- wall-clock timeout;
- token and cost budget;
- allowed tool set;
- argument and result-size limits;
- terminal conditions for success, denial, insufficient evidence, and failure.

An agent must be able to stop without producing a successful answer.

### Work it out first

Limits:

`max_steps=8`, `max_tool_calls=3`, `budget=$0.05`, `timeout=30s`

After three searches with no supporting evidence, route to `insufficient_evidence` rather than search indefinitely.

### Notebook bridge

The lab tests successful, insufficient, denied, timeout, and budget-exhausted endings.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
if state["tool_calls"] >= 3 or state["cost"] >= 0.05:
    return Command(update={"status": "budget_exhausted"}, goto=END)

Expected output:

```text
A terminal state reached before the execution exceeds its bounds.
```


## 11. A Minimal LangGraph Is Explicit Python

Build order:

1. define typed state;
2. implement small node functions;
3. add nodes;
4. add fixed and conditional edges;
5. compile with runtime controls;
6. invoke with initial state and configuration;
7. inspect final state and trace.

Compilation checks graph structure. It does not prove node logic or safety.

### Work it out first

`START -> retrieve -> route`

- documents -> answer -> END
- no documents -> insufficient -> END

There is no loop because one retrieval is enough for this workflow.

### Notebook bridge

This prepares learners to read the quickstart workflow and agent graphs.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
builder = StateGraph(State)
builder.add_node("retrieve", retrieve)
builder.add_node("answer", answer)
builder.add_edge(START, "retrieve")
builder.add_conditional_edges("retrieve", route)
graph = builder.compile()

Expected output:

```text
A compiled graph whose allowed paths match the declared edges.
```


## 12. Guided Lab: Build a Bounded Research Workflow

Build a policy-research graph with:

1. typed state;
2. validate, route, retrieve, answer, and finish nodes;
3. one deterministic route;
4. one model-selected read-only tool;
5. argument validation and authorization;
6. observations stored with provenance;
7. checkpoint persistence;
8. bounded retries for transient search failure;
9. human approval before a mock publish action;
10. maximum steps, calls, time, and cost;
11. complete tracing;
12. tests for success, insufficient evidence, denial, timeout, and rejected approval.

### Work it out first

Trace:

`validate -> retrieve -> answer -> review -> rejected -> END`

The final status is `rejected`; no publish side effect occurs.

### Notebook bridge

Complete `29.langgraph-quickstart.ipynb`; use `18.langgraph-basics.ipynb` only as an optional secondary exercise.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
result = graph.invoke(
    {"question": question, "tool_calls": 0, "status": "started"},
    {"configurable": {"thread_id": request_id}, "recursion_limit": 10},
)

Expected output:

```text
A typed final or interrupted state within the declared execution limit.
```


## Guided lab

Build a policy-research graph with:

1. typed state;
2. validate, route, retrieve, answer, and finish nodes;
3. one deterministic route;
4. one model-selected read-only tool;
5. argument validation and authorization;
6. observations stored with provenance;
7. checkpoint persistence;
8. bounded retries for transient search failure;
9. human approval before a mock publish action;
10. maximum steps, calls, time, and cost;
11. complete tracing;
12. tests for success, insufficient evidence, denial, timeout, and rejected approval.

### Reference result

Trace:

`validate -> retrieve -> answer -> review -> rejected -> END`

The final status is `rejected`; no publish side effect occurs.


In [ ]:
# Guided lab workspace: Week 17
# Add only the imports needed for the current step.

# TODO 1: Prepare the smallest valid input.

# TODO 2: Apply the concept taught in this lesson.

# TODO 3: Display inspectable intermediate evidence.

# TODO 4: Compare the result with a hand calculation or stated requirement.

## Weekly deliverable

Submit the completed guided lab with:

- your prediction before execution;
- intermediate values, shapes, metrics, or traces;
- one failed assumption and its correction;
- a plain-English explanation of the result;
- the source notebook section you are now ready to complete.


## Sources and source notebooks

- <https://docs.langchain.com/oss/python/langgraph/overview>
- <https://github.com/curiousily/AI-Bootcamp/blob/master/29.langgraph-quickstart.ipynb>
- <https://docs.langchain.com/oss/python/langgraph/graph-api>
- <https://docs.langchain.com/oss/python/langgraph/thinking-in-langgraph>
- <https://docs.langchain.com/oss/python/langgraph/use-graph-api>
- <https://docs.langchain.com/oss/python/langgraph/agentic-rag>
- <https://modelcontextprotocol.io/docs/concepts/tools>
- <https://docs.langchain.com/oss/python/langgraph/persistence>
- <https://docs.langchain.com/oss/python/langgraph/fault-tolerance>
- <https://docs.aws.amazon.com/prescriptive-guidance/latest/cloud-design-patterns/retry-backoff.html>
- <https://docs.langchain.com/oss/python/langgraph/interrupts>
- <https://docs.langchain.com/oss/python/langgraph/use-graph-api#create-and-control-loops>
- <https://github.com/curiousily/AI-Bootcamp/blob/master/18.langgraph-basics.ipynb>